# Conexión con el sensor

En este notebook nos conectamos al WT9011DCL por Bluetooth, vemos cómo llegan los datos en bytes y los convertimos a unidades físicas: aceleración en g, velocidad angular en °/s y ángulos en grados. Al final leemos la temperatura y la batería del sensor.

El sensor debe estar encendido y sin conexión con otro equipo.

## Parámetros

Cada equipo escribe la dirección de su sensor, la que se obtuvo en `00_entorno`. Los UUID identifican los dos canales que usaremos: por FFE4 el sensor envía datos y por FFE9 recibe comandos.

In [ ]:
DIRECCION = "FA:81:12:D8:44:D0"   # cambiar por la dirección del sensor del equipo

UUID_DATOS = "0000ffe4-0000-1000-8000-00805f9a34fb"      # el sensor envía (notify)
UUID_COMANDOS = "0000ffe9-0000-1000-8000-00805f9a34fb"   # el sensor recibe (write)

## Conexión

La conexión queda abierta mientras se trabaja con el resto del notebook. Al final hay una celda para desconectar; si no se ejecuta, el sensor sigue ocupado y no aparece en otros equipos.

In [ ]:
from bleak import BleakClient

cliente = BleakClient(DIRECCION, timeout=20)
await cliente.connect()
print("Conectado:", cliente.is_connected)

## Servicios y características

En BLE la información se organiza en servicios, y cada servicio agrupa características. Las propiedades de una característica indican qué se puede hacer con ella: read (leer), write (escribir) o notify (el dispositivo avisa cada vez que tiene datos nuevos, sin que se los pidamos).

In [ ]:
for servicio in cliente.services:
    print(servicio.uuid, servicio.description)
    for c in servicio.characteristics:
        print("   ", c.uuid, ", ".join(c.properties))

Los dos primeros servicios son estándar en cualquier dispositivo BLE. El tercero, FFE5, es del fabricante y contiene las dos características que nos interesan: FFE4 con notify y FFE9 con write.

## Datos crudos

Nos suscribimos a FFE4 durante un segundo. Cada vez que llega una notificación, la función `al_recibir` guarda los bytes tal como llegaron.

In [ ]:
import asyncio

crudos = []

def al_recibir(_, datos):
    crudos.append(bytes(datos))

await cliente.start_notify(UUID_DATOS, al_recibir)
await asyncio.sleep(1)
await cliente.stop_notify(UUID_DATOS)

print(f"Notificaciones recibidas: {len(crudos)}")
print(f"Bytes por notificación:   {sorted(set(len(c) for c in crudos))}")
print()
print(crudos[0].hex(" "))

Cada notificación trae varios paquetes pegados. Los paquetes de datos miden 20 bytes y empiezan con 55 61, así que una notificación de 80 bytes contiene 4 paquetes. Un paquete también puede quedar partido entre dos notificaciones, por eso se juntan todos los bytes en un buffer y de ahí se cortan los paquetes completos.

Para encontrar el inicio de cada paquete se busca el byte 55. Como ese valor también puede aparecer dentro de los datos, se revisa además que el siguiente byte sea un tipo válido (61 o 71).

In [ ]:
def separar_paquetes(buffer):
    """Extrae del buffer los paquetes completos de 20 bytes. Los bytes sobrantes se quedan en el buffer."""
    paquetes = []
    while len(buffer) >= 20:
        if buffer[0] == 0x55 and buffer[1] in (0x61, 0x71):
            paquetes.append(bytes(buffer[:20]))
            del buffer[:20]
        else:
            del buffer[0]
    return paquetes

buffer = bytearray(b"".join(crudos))
paquetes = separar_paquetes(buffer)

print(f"Paquetes: {len(paquetes)}, bytes sobrantes: {len(buffer)}\n")
for p in paquetes[:4]:
    print(p.hex(" "))

## Estructura de un paquete

| Bytes | Contenido |
|---|---|
| 0 | encabezado, siempre 0x55 |
| 1 | tipo de paquete: 0x61 datos, 0x71 respuesta a un comando |
| 2 a 7 | aceleración en x, y, z |
| 8 a 13 | velocidad angular en x, y, z |
| 14 a 19 | ángulos roll, pitch y yaw |

Cada magnitud ocupa 2 bytes: primero el byte bajo y después el alto (little endian). Juntos forman un entero con signo de 16 bits, entre -32768 y 32767. Para pasarlo a unidades físicas se divide entre 32768 y se multiplica por el rango de medición:

| Magnitud | Rango | Conversión |
|---|---|---|
| aceleración | ±16 g | valor / 32768 × 16 |
| velocidad angular | ±2000 °/s | valor / 32768 × 2000 |
| ángulo | ±180° | valor / 32768 × 180 |

## Decodificación de un valor, paso a paso

Tomamos la aceleración en x del primer paquete, que está en los bytes 2 y 3.

In [ ]:
p = paquetes[0]
bajo, alto = p[2], p[3]
print(f"Byte bajo: 0x{bajo:02X} = {bajo}")
print(f"Byte alto: 0x{alto:02X} = {alto}")

valor = (alto << 8) | bajo     # el byte alto se recorre 8 bits a la izquierda
print(f"Entero sin signo: {valor}")

if valor >= 32768:             # complemento a dos: la mitad superior corresponde a negativos
    valor -= 65536
print(f"Entero con signo: {valor}")

ax = valor / 32768 * 16
print(f"Aceleración en x: {ax:.3f} g")

El módulo `struct` hace lo mismo con los 9 valores a la vez. El formato `<9h` indica little endian (`<`) y 9 enteros de 16 bits con signo (`h`).

In [ ]:
import struct

NOMBRES = ["ax", "ay", "az", "gx", "gy", "gz", "roll", "pitch", "yaw"]
ESCALAS = [16] * 3 + [2000] * 3 + [180] * 3

def decodificar(paquete):
    """Convierte un paquete 0x61 en un diccionario con aceleración [g], velocidad angular [°/s] y ángulos [°]."""
    enteros = struct.unpack("<9h", paquete[2:20])
    return {n: v / 32768 * e for n, v, e in zip(NOMBRES, enteros, ESCALAS)}

for nombre, valor in decodificar(paquetes[0]).items():
    print(f"{nombre:<6}{valor:9.3f}")

## Lectura continua

Ahora se combinan los pasos anteriores: durante 3 segundos se reciben notificaciones, se separan los paquetes y se decodifican. Para esta prueba el sensor debe estar quieto sobre la mesa.

In [ ]:
import pandas as pd

buffer = bytearray()
lecturas = []

def al_recibir(_, datos):
    buffer.extend(datos)
    for p in separar_paquetes(buffer):
        if p[1] == 0x61:
            lecturas.append(decodificar(p))

DURACION = 3   # segundos

await cliente.start_notify(UUID_DATOS, al_recibir)
await asyncio.sleep(DURACION)
await cliente.stop_notify(UUID_DATOS)

df = pd.DataFrame(lecturas)
print(f"Paquetes: {len(df)}, alrededor de {len(df) / DURACION:.0f} por segundo\n")
df.round(3).head(10)

Con el sensor en reposo la única aceleración que mide es la de la gravedad, así que la magnitud del vector de aceleración debe estar cerca de 1 g sin importar cómo esté orientado, y la velocidad angular debe estar cerca de 0. Es una forma rápida de comprobar que la decodificación es correcta.

In [ ]:
import numpy as np

magnitud = np.sqrt(df.ax**2 + df.ay**2 + df.az**2)
print(f"Magnitud de la aceleración, promedio: {magnitud.mean():.3f} g")
print(f"Velocidad angular, valor máximo:      {df[['gx', 'gy', 'gz']].abs().max().max():.2f} °/s")

El número de paquetes por segundo es la frecuencia de muestreo con la que trabaja el sensor. En el siguiente notebook veremos cómo cambiarla y por qué el valor medido no coincide exactamente con el configurado.

## Lectura de registros: temperatura y batería

El sensor guarda su configuración y algunas mediciones adicionales en registros numerados. Para leer uno se envía por FFE9 el comando `FF AA 27`, el número de registro y `00`. El sensor contesta con un paquete de tipo 0x71 que trae ese registro y los siete siguientes, cada uno como entero de 16 bits.

Aquí se usan dos registros: 0x40, temperatura en centésimas de °C, y 0x64, voltaje de la batería en centésimas de volt.

In [ ]:
respuestas = {}

def al_recibir(_, datos):
    buffer.extend(datos)
    for p in separar_paquetes(buffer):
        if p[1] == 0x71:
            registro = p[2] | (p[3] << 8)
            respuestas[registro] = struct.unpack("<8h", p[4:20])

async def leer_registro(registro):
    """Pide al sensor el valor de un registro y espera la respuesta hasta 1 s."""
    respuestas.pop(registro, None)
    comando = bytes([0xFF, 0xAA, 0x27, registro, 0x00])
    await cliente.write_gatt_char(UUID_COMANDOS, comando, response=False)
    for _ in range(20):
        await asyncio.sleep(0.05)
        if registro in respuestas:
            return respuestas[registro][0]
    raise TimeoutError(f"El sensor no respondió la lectura del registro 0x{registro:02X}")

buffer = bytearray()
await cliente.start_notify(UUID_DATOS, al_recibir)
temperatura = await leer_registro(0x40) / 100
voltaje = await leer_registro(0x64) / 100
await cliente.stop_notify(UUID_DATOS)

print(f"Temperatura: {temperatura:.1f} °C")
print(f"Batería:     {voltaje:.2f} V")

El voltaje de una batería de litio no baja de forma proporcional a la carga, por eso el porcentaje se estima con una tabla. Estos son los valores que usa el fabricante.

In [ ]:
def porcentaje_bateria(voltaje):
    tabla = [(3.96, 100), (3.93, 90), (3.87, 75), (3.82, 60), (3.79, 50), (3.77, 40),
             (3.73, 30), (3.70, 20), (3.68, 15), (3.50, 10), (3.40, 5)]
    for umbral, porcentaje in tabla:
        if voltaje > umbral:
            return porcentaje
    return 0

print(f"Carga aproximada: {porcentaje_bateria(voltaje)} %")

## Desconexión

In [ ]:
await cliente.disconnect()
print("Conectado:", cliente.is_connected)

En el siguiente notebook, `02_adquisicion`, las funciones `separar_paquetes`, `decodificar` y `leer_registro` se reúnen en el archivo `wt9011.py` para usarlas sin repetir el código. Ahí se configura la frecuencia de muestreo y se graban las primeras señales.